In [2]:
!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/nyc_tripdata_2024_sample_4M.csv
from pyspark.sql import SparkSession
spark = (
SparkSession.builder
.appName("ExerciciosPySpark")
.master("local[*]")
.getOrCreate()
)
df = spark.read.csv("nyc_tripdata_2024_sample_4M.csv", header=True,
inferSchema=True)

###Questão 1

In [12]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [13]:
df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-10-01 00:59:55|  2024-10-01 01:02:24|              1|          0.5|         1|                 N|         230|         161|           1|        5.1|  3.5|    0.5|       2.

In [14]:
df.count()

4118743

###questão 2

In [15]:
df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "payment_type"
).show(5)

+--------+--------------------+-------------+-----------+------------+
|VendorID|tpep_pickup_datetime|trip_distance|fare_amount|payment_type|
+--------+--------------------+-------------+-----------+------------+
|       1| 2024-10-01 00:59:55|          0.5|        5.1|           1|
|       1| 2024-10-01 00:08:59|         20.6|       76.5|           2|
|       2| 2024-10-01 00:18:38|         7.42|       33.1|           4|
|       2| 2024-10-01 00:20:06|        19.96|       70.0|           1|
|       1| 2024-10-01 00:09:02|          2.6|       15.6|           1|
+--------+--------------------+-------------+-----------+------------+
only showing top 5 rows


###questão 3

In [16]:
corridas_filtradas = df.filter(
    (df.trip_distance > 5) &
    (df.passenger_count >= 3)
)

print("Quantidade de corridas:", corridas_filtradas.count())

Quantidade de corridas: 50665


###questão 4

Com inferSchema=True, o Spark analisa os dados para identificar automaticamente o tipo de cada coluna. Essa abordagem é mais prática e rápida de configurar, mas pode consumir mais recursos e inferir tipos incorretamente. Já o StructType/StructField permite definir os tipos manualmente, oferecendo maior controle, consistência e eficiência. Em arquivos com milhões de linhas, o schema manual tende a ser mais adequado.

###questão 5

In [17]:
from pyspark.sql import functions as F

resultado = df.groupBy("payment_type").agg(
    F.count("*").alias("quantidade_corridas"),
    F.sum("total_amount").alias("receita_total")
).orderBy("receita_total", ascending=False)

resultado.show()

+------------+-------------------+--------------------+
|payment_type|quantidade_corridas|       receita_total|
+------------+-------------------+--------------------+
|           1|            3045849| 9.116799616010016E7|
|           2|             553536|1.2987084559999354E7|
|           0|             410746|1.0123049400000528E7|
|           3|              29100|  220775.24999999974|
|           4|              79511|  133192.01999999984|
|           5|                  1|                62.0|
+------------+-------------------+--------------------+



###questão 6

In [18]:
resultado = df.withColumn(
    "hora_embarque",
    df.tpep_pickup_datetime.cast("timestamp").substr(12, 2).cast("int")
).groupBy("hora_embarque").agg(
    {"fare_amount": "avg", "trip_distance": "avg"}
).orderBy("hora_embarque")

resultado.show()

+-------------+------------------+------------------+
|hora_embarque|  avg(fare_amount)|avg(trip_distance)|
+-------------+------------------+------------------+
|            0| 19.72867660335703| 5.130178643081671|
|            1| 17.54847894641617| 3.739999483030465|
|            2|16.426538461538446| 4.542445678033303|
|            3| 17.24064640950263| 3.401675265462839|
|            4| 22.33585451861572|11.412191651631977|
|            5|26.226564065583663| 23.33988120540463|
|            6|21.931859821807333|14.540589393296582|
|            7| 19.33026953083623|11.087329050022918|
|            8|18.511148615351107| 8.533842520592145|
|            9|18.400291333656767| 5.604490502277248|
|           10| 18.56454835768904|4.5114450807098905|
|           11|18.851552824117647| 4.075729557436569|
|           12|19.207714073999153| 4.468694683646846|
|           13| 19.95819012628161| 5.262006204074442|
|           14|20.604332332373676| 4.622973056355365|
|           15| 20.757107834

###questão 7

Transformações, como select(), filter() e groupBy(), apenas definem o que o Spark deve fazer, sem executar imediatamente. Já ações, como show() e count(), realmente executam o processamento. O Spark usa avaliação preguiçosa porque espera uma ação para executar as transformações, podendo assim otimizar o processamento e evitar operações desnecessárias, o que melhora o desempenho.

###questão 8

In [19]:
resultado = df.filter(
    df.total_amount > 0
).withColumn(
    "percentual_gorjeta",
    (df.tip_amount / df.total_amount) * 100
).select(
    "VendorID",
    "total_amount",
    "tip_amount",
    "percentual_gorjeta"
).orderBy(
    "percentual_gorjeta",
    ascending=False
)

resultado.show(10)

+--------+------------+----------+------------------+
|VendorID|total_amount|tip_amount|percentual_gorjeta|
+--------+------------+----------+------------------+
|       2|        1.63|      5.27| 323.3128834355828|
|       2|        2.07|      3.68| 177.7777777777778|
|       2|         1.6|      2.82|176.24999999999997|
|       2|        2.33|      3.72|159.65665236051504|
|       2|        2.54|      3.76|148.03149606299212|
|       2|        3.76|      3.96|105.31914893617022|
|       2|        39.7|      40.0|100.75566750629723|
|       2|        0.08|      0.08|             100.0|
|       1|       197.0|     196.0| 99.49238578680203|
|       1|       150.0|     149.0| 99.33333333333333|
+--------+------------+----------+------------------+
only showing top 10 rows


###questão 9

In [21]:
!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/taxi_zone_lookup.csv
zonas = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)
zonas.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [22]:
resultado = df.join(
    zonas,
    df.PULocationID == zonas.LocationID
).groupBy(
    "Borough"
).count().orderBy(
    "count",
    ascending=False
)

resultado.show()

+-------------+-------+
|      Borough|  count|
+-------------+-------+
|    Manhattan|3641752|
|       Queens| 388736|
|     Brooklyn|  60200|
|        Bronx|  12702|
|      Unknown|  12172|
|          N/A|   2421|
|          EWR|    565|
|Staten Island|    195|
+-------------+-------+



###questão 10

O count() apenas percorre os dados para contabilizar as linhas, enquanto o groupBy() precisa reorganizar os dados entre as partições, processo chamado shuffle. Por isso, o groupBy() tende a ser mais demorado e consumir mais recursos, pois envolve movimentação e redistribuição dos dados, enquanto operações simples como count(), filtros e seleção de colunas geralmente exigem menos processamento.